# TR8D Laya GPU training and 85% selective-accuracy gate

This notebook trains a reproducible head-only baseline on a GPU. It does not promise 85% overall market-direction accuracy. It promotes a checkpoint only if calibration-selected high-confidence decisions reach at least 85% accuracy at 10%+ coverage on a later untouched test split.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing'
print(torch.cuda.get_device_name(0))

In [ ]:
!git clone --branch agent/transactional-paper-execution https://github.com/raj291/tr8d.git /content/tr8d
%cd /content/tr8d
!python -m pip install -q -e '.[agent,dev]'

Upload a normalized point-in-time CSV named `prices.csv`. Required columns are `symbol,date,open,close`; `available_at` is strongly recommended. Keep the final chronological test period sealed.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert 'prices.csv' in uploaded
!python -m tr8d export-laya-dataset prices.csv --output /content/laya-training
!cat /content/laya-training/manifest.json

In [ ]:
!python -m tr8d train-laya --dataset /content/laya-training --output /content/laya-tr8d --device cuda --epochs 8 --patience 2 --batch-size 16 --target-accuracy 0.85 --minimum-coverage 0.10 --minimum-test-examples 500

In [ ]:
!python -m tr8d evaluate-laya --dataset /content/laya-training --model /content/laya-tr8d --device cuda --batch-size 16 --target-accuracy 0.85 --minimum-coverage 0.10 --minimum-test-examples 500 --write-policy --output /content/laya-quality-report.json
import json
report = json.load(open('/content/laya-quality-report.json'))
print(json.dumps(report['promotion_gate'], indent=2))
assert report['promoted'], 'Checkpoint is not safe to promote; do not weaken the gate'

## Full RLCD next step

The baseline above freezes the encoder. For full encoder + head training, use Laya's official dual-T4 RLCD notebook and replace its dataset preprocessing cell with `/content/laya-training/train_rlcd.jsonl`. Keep TR8D's calibration and test files separate. Official notebook: https://github.com/NandhaKishorM/laya/blob/main/notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb

In [ ]:
!cd /content && zip -qr laya-tr8d-approved.zip laya-tr8d laya-quality-report.json
from google.colab import files
files.download('/content/laya-tr8d-approved.zip')